In [ ]:
# Databricks notebook source
# notebooks/01_bronze/bronze_schema_repair
#
# For Each child task — receives one table per iteration.
# Recovers values from _rescued_data via MERGE INTO.
# Parameter injected by Workflow For Each task as widget "input".

import sys
import json
sys.path.insert(0, "/Workspace/Repos/tarun220505@hotmail.com/credit-risk-fraud-lakehouse/src")

from config.pipeline_config import PipelineConfig
from utils.logger import PipelineLogger
from bronze.repair import BronzeRepair

# ---------------------------------------------------------------------------
# Parameter from For Each task
# Default is empty — must be injected by Workflow
# ---------------------------------------------------------------------------
dbutils.widgets.removeAll()
dbutils.widgets.text("input", "")

raw_input = dbutils.widgets.get("input").strip()
if not raw_input:
    raise ValueError(
        "No input parameter received. "
        "This notebook must be run via Workflow For Each task."
    )

item         = json.loads(raw_input)
table_name   = item["table"]
primary_key  = item["primary_key"]
rescued_cols = [c.strip() for c in item["rescued_cols"].split(",")]

# ---------------------------------------------------------------------------
# Run repair
# ---------------------------------------------------------------------------
config = PipelineConfig()
logger = PipelineLogger(
    spark,
    config,
    task_name=f"bronze_repair_{table_name}",
)
repair = BronzeRepair(spark, config, logger)

result = repair.repair_table(
    table_name   = table_name,
    primary_key  = primary_key,
    rescued_cols = rescued_cols,
)

dbutils.notebook.exit(json.dumps({
    "table":               result.table_name,
    "rows_rescued_before": result.rows_rescued_before,
    "rows_rescued_after":  result.rows_rescued_after,
    "rows_repaired":       result.rows_repaired,
    "status":              result.status,
}))